# 1 卷积神经网络
卷积神经网络（convolutional neural network，CNN）是一类强大的、为处理图像数据而设计的神经网络。基于卷积神经网络架构的模型在计算机视觉领域已经占主导地位。
## 1.1 从全连接层到卷积

多层感知机是否适合处理表格数据，行对应样本，列对应特征。寻找的模式可能涉及特征之间的交互，但是不能预先假设任何与特征相关的先验结构。此时多层感知机可能是最好的选择，然而对于高维感知数据，这种缺少结构的网络可能会不实用。
### 不变性

想从一张图片中找到某个物体，合理的假设是：无论哪种方法找到这个物体，都应该和物体的位置无关。卷积神经网络正是将空间不变性（spatial invariance）的这一概念系统化，从而基于这个模型使用较少的参数来学习有用的表示。
- 平移不变性（translation invariance）：不管检测对象出现在图像中的哪个位置，神经网络的前面几层应该对相同的图像区域具有相似的反应，即平移不变性。
- 局部性（locality）：神经网络的前面几层应该只探索输入图像中的局部区域，而不过度在意相隔较远居于的关系。

### 多层感知机的限制

多层感知机的输入是二维图像$\bf X $，其隐藏表示$\bf H$在数学上是一个矩阵，代码中表示为二维张量。$\bf X、H$具有相同的形状。使用$\mathbf [X]_{ij}$和$\mathbf [H]_{ij}$分别表示输入图像和隐藏表示中位置（i,j）处的像素。为了使每个隐藏神经元都能接收到每个输入像素的信息，将权重矩阵替换为为四阶权重张量$W$。$\bf U$包含偏置参数，全连接层形式化表示为
$$
\begin{aligned}
\mathbf [H]_{i,j}&=\mathbf [U]_{i,j} + \sum_{k}\sum_{l}[W]_{i,j,k,l}\mathbf[X]_{k,l}\\
                 &=\mathbf [U]_{i,j} + \sum_{a}\sum_{b}[W]_{i,j,a,b}\mathbf[X]_{i+a,j+b}
\end{aligned}
$$
从W到V的变化只是形式上的变化，只需重新索引下标$(k, l)=(k=i+1, l=j+b)$，索引a和索引b通过正偏移和负偏移之间移动覆盖了整个图像。对于隐藏表示中任意给定位置$(i,j)$处的像素值$[\mathbf H]_{i,j}$，可以通过在x中以(i,j)为重心对像素进行加权求和得到，加权使用的权重为$[V]_{i,j,a,b}$

#### 平移不变性

引用上述第一个原则，平移不变性。意味着检测对象在输入$\bf X$中的平移，应该仅导致隐藏表示$\bf H$中的平移。也就是说，V和$\bf U$实际上不依赖（i，j）的值，即$\mathbf[V]_{i,j,a,b}=\mathbf[V]_{a,b}$。并且U是一个常数。简化$bf H$定义为：
$$
[\mathbf H]_{i,j}=u+\sum_a\sum_b[\mathbf V]_{a,b}[\mathbf X]_{i+a,j+b}
$$
这就是卷积（convolution）。使用系数$[\mathbf H]_{i,j}$对位置(i,j)附近的像素(i+a,j+b)进行加权得到$\mathbf [H]_{i,j}$。注意系数大量减少

#### 局部性

引用第二个原则，局部性。为了收集训练参数$[\mathbf H]_{i,j}$的相关信息。不应偏离距(i,j)很远的地方。在$|a|\gt\Delta、|b|\gt\Delta$的范围之外，可以设置$\mathbf[V]_{a,b}=0$。因此，将$\mathbf [H]_{i,j}$重写为
$$
[H]_{i,j}=u+\sum_{a=-\Delta}^\Delta\sum_{b=-\Delta}^\Delta[\mathbf V]_{a,b}[\mathbf X]_{i+a,j+b}
$$

简而言之，上式就是一个卷积层（convolutional layer），而卷积神经网络是包含卷积层的一类特殊的神经网络。$\bf V$称之为卷积核（convolutional kernel）或者滤波器（filter），亦或简单称之为卷积层的权重，通常该权重是可学习的参数。

### 卷积

数学上，两个函数$(f,g):\R^d\to\R $之间的卷积定义为
$$
(f*g)(\mathbf x)=\int f(\mathbf z)g(\mathbf{x-z})d\mathbf z
$$
即把一个函数“翻转”并移位$\mathbf x$时，测量f和g之间的重叠。当为离散对象时，积分就变成求和。对于二维张量，f的索引$(a,b)$和g的索引$i-a,j-b$的对应加和为
$$
(f*g)(i,j)=\sum_a\sum_bf(a,b)g(i-a,j-b)
$$
#### 通道

图像一般包含三个通道/原色（红、绿、蓝）。图像实际上是一个由高度、宽度和颜色组成的三维张量。将X索引为$[X]_{i,j,k}$。卷积相应的调整为$[V]_{a,b,c}$。
对于每一个空间位置，
